# Customer Segmentation ML — Complete Pipeline

This notebook walks through all 13 steps of the customer segmentation ML pipeline:
1. Problem Definition & Data Generation
2. Data Collection & Exploration
3. Data Preparation & Preprocessing
4. Exploratory Data Analysis
5. Label Construction
6. Train/Test Split
7. Model Selection & Training
8. Hyperparameter Tuning
9. Model Evaluation
10. Feature Importance & Explainability
11. Model Validation
12. Production Deployment
13. Monitoring & Maintenance

## Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete!')

## Step 1 & 2: Problem Definition & Data Generation

In [ ]:
from importlib import import_module
dg = import_module('01_data_generation')
df = dg.generate_customer_data(n_samples=5000)
print(f'Generated {len(df)} customer records')
print('\nClass distribution:')
print(df['segment'].value_counts())
df.head()

In [ ]:
df.describe()

## Step 3: Data Preprocessing

In [ ]:
# Run preprocessing pipeline
import subprocess
result = subprocess.run(['python', '../src/02_data_preprocessing.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 4: Exploratory Data Analysis

In [ ]:
processed_df = pd.read_csv('../data/customer_data_processed.csv')
print(f'Processed data shape: {processed_df.shape}')

# Segment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = processed_df['segment'].value_counts()
colors = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 'High Risk': '#e74c3c'}
bar_colors = [colors.get(c, 'gray') for c in counts.index]
axes[0].bar(counts.index, counts.values, color=bar_colors)
axes[0].set_title('Class Distribution')
counts.plot.pie(ax=axes[1], colors=bar_colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = [
    'age', 'income', 'premium_amount', 'num_missed_payments_12m',
    'avg_payment_delay_days', 'credit_score', 'payment_consistency_score',
    'payment_risk_score'
]
numerical_cols = [c for c in numerical_cols if c in processed_df.columns]
plt.figure(figsize=(12, 9))
corr = processed_df[numerical_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for key payment features by segment
key_features = ['num_missed_payments_12m', 'avg_payment_delay_days', 'credit_score', 'payment_consistency_score']
key_features = [c for c in key_features if c in processed_df.columns]
fig, axes = plt.subplots(1, len(key_features), figsize=(16, 5))
order = [s for s in ['Low Risk', 'Medium Risk', 'High Risk'] if s in processed_df['segment'].unique()]
for ax, feat in zip(axes, key_features):
    data = [processed_df[processed_df['segment'] == seg][feat].dropna().values for seg in order]
    bp = ax.boxplot(data, labels=order, patch_artist=True)
    for patch, seg in zip(bp['boxes'], order):
        patch.set_facecolor(colors.get(seg, 'gray'))
        patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=11)
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('Key Features by Segment', fontsize=14)
plt.tight_layout()
plt.show()

## Steps 7–8: Model Training & Hyperparameter Tuning

In [ ]:
import subprocess
print('Training models... (this may take a few minutes)')
result = subprocess.run(['python', '../src/04_model_training.py'], capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## Step 9: Model Evaluation

In [ ]:
import json
import pickle

with open('../data/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

print('Class names:', splits['class_names'])
print('Test set shape:', splits['X_test'].shape)

# Load evaluation metrics if available
metrics_path = '../reports/evaluation_metrics.json'
import os
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics_df = pd.DataFrame(metrics).T
    print('\nModel comparison:')
    print(metrics_df.to_string())
else:
    print('Run 05_model_evaluation.py to generate metrics')

## Step 10: Feature Importance

In [ ]:
import pickle
import os

rf_path = '../models/random_forest.pkl'
preprocessor_path = '../models/preprocessor.pkl'

if os.path.exists(rf_path) and os.path.exists(preprocessor_path):
    with open(rf_path, 'rb') as f:
        rf_model = pickle.load(f)
    with open(preprocessor_path, 'rb') as f:
        prep_data = pickle.load(f)
    
    preprocessor = prep_data['preprocessor']
    feature_names = list(preprocessor.get_feature_names_out())
    
    importances = rf_model.feature_importances_
    fi_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)[:15]
    
    plt.figure(figsize=(10, 6))
    fi_series.plot.barh(color='steelblue', alpha=0.8)
    plt.title('Random Forest – Top 15 Feature Importances')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('Train models first (04_model_training.py)')

## Step 12: API Demo

In [ ]:
# Example of what the API call would look like (run app.py first)
sample_customer = {
    'age': 45,
    'income': 72000,
    'premium_amount': 1800,
    'policy_tenure_months': 36,
    'num_missed_payments_12m': 0,
    'avg_payment_delay_days': 3,
    'claims_frequency': 0,
    'credit_score': 780,
    'payment_consistency_score': 0.95,
    'account_age_months': 72,
    'policy_type': 'Life',
    'location': 'Suburban'
}

print('Sample customer (expected: Low Risk):')
for k, v in sample_customer.items():
    print(f'  {k}: {v}')

print('\nTo get a prediction, run:')
print('  python src/app.py')
print('  curl -X POST http://localhost:5000/predict -H "Content-Type: application/json" -d \'{...}\'')

## Summary

This pipeline demonstrates a complete ML workflow:

1. ✅ **Data Generation** – 5000 synthetic customers with realistic distributions
2. ✅ **Preprocessing** – Missing values, outliers, feature engineering, scaling
3. ✅ **EDA** – Distributions, correlations, segment-wise analysis
4. ✅ **Modeling** – 4 algorithms with 5-fold CV and GridSearchCV
5. ✅ **Evaluation** – Accuracy, F1, AUC-ROC, confusion matrices
6. ✅ **Explainability** – SHAP values, feature importance, permutation importance
7. ✅ **Deployment** – Flask REST API with 6 endpoints
8. ✅ **Monitoring** – Drift detection (KS test + PSI) + automated retraining